<a href="https://colab.research.google.com/github/fborrasumh/bioestadistica/blob/main/cuadernos/Tema_090_Programacion_asistida_por_IA_y_mejora_recursiva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# <font color='steelblue'>Tema 90. Programación asistida por inteligencia artificial: ciclo de mejora recursiva de generación, verificación y corrección</font>

**Bioestadística (1964) · Grado en Medicina · 2.º curso · Curso 2027-2028**

**Módulo 3. Procesado inicial de datos**

**Autor:** Fernando Borrás Rocher · Universidad Miguel Hernández de Elche · ORCID [0000-0002-5519-4573](https://orcid.org/0000-0002-5519-4573)

**Duración:** 50 minutos de clase, más el estudio de los apartados de ampliación

**Práctica:** [unidad 90 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)

**Requisito:** Temas 50 (funciones y casos de prueba) y 80 (API). Sin clave, el cuaderno funciona con respuestas simuladas.

**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es). Todos los pacientes y datos de este cuaderno son sintéticos.

*Antes de trabajar, guarda tu copia: **Archivo → Guardar una copia en Drive**.*

## <font color='steelblue'>Configuración del cuaderno</font>

Ejecuta esta celda al empezar. Instala el módulo de preguntas interactivas y prepara el cuaderno.

**Para dar la clase con el cuaderno:** usa **Entorno de ejecución → Ejecutar todo** antes de empezar, para que se dibujen los gráficos. Las celdas con borde azul son las diapositivas; el desarrollo de cada bloque está debajo y se puede plegar con la flecha de su título. El índice (☰, a la izquierda) sirve para saltar de un bloque a otro.

In [ ]:
#@title Ejecutar configuración
%pip install -q jupyterquiz==2.9.6.4
from jupyterquiz import display_quiz
print("Cuaderno configurado")

## <font color='steelblue'>Objetivos de aprendizaje</font>

Al terminar esta sesión serás capaz de:

1. Describir el ciclo de **mejora recursiva**: generar, ejecutar, verificar y corregir.
2. Escribir una **especificación** y sus **casos de prueba** antes de pedir código a un modelo de IA.
3. Programar un bucle que pida código al modelo, lo pruebe y le devuelva los fallos hasta que las pruebas se superen o se agote el número de intentos.
4. Registrar cada iteración del ciclo para poder analizarla después.
5. Explicar por qué superar las pruebas no garantiza que el código sea correcto (ley de Goodhart).
6. Aplicar las medidas de seguridad necesarias antes de ejecutar código generado por IA.

## <font color='steelblue'>Plan de la sesión</font>

| Bloque | Minutos |
| --- | --- |
| 1. Caso clínico de partida | 5 |
| 2. Conceptos clave | 10 |
| 3. Demostración: un ciclo de mejora recursiva | 20 |
| 4. Práctica en la app Bioestadística | 10 |
| 5. Autoevaluación | 5 |

## <font color='steelblue'>1. Caso clínico de partida (5 min)</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">«Funciona» no es lo mismo que «es correcto»</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_090_curb.png" width="780" alt="«Funciona» no es lo mismo que «es correcto»"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>El código del asistente se ejecuta sin errores… y se equivoca justo en las fronteras.</li><li>Solo lo descubre quien tiene casos de prueba escritos a partir de la definición clínica.</li></ul></div>

#### Desarrollo para el estudio

El servicio de Urgencias quiere que la historia clínica calcule automáticamente la escala **CURB-65** de gravedad de la neumonía adquirida en la comunidad. La escala suma un punto por cada criterio:

| Criterio | Punto si… |
| --- | --- |
| **C**onfusión | Desorientación de nueva aparición |
| **U**rea | Urea > 7 mmol/L |
| Frecuencia **R**espiratoria | ≥ 30 rpm |
| Presión arterial (**B**lood pressure) | PAS < 90 mmHg **o** PAD ≤ 60 mmHg |
| Edad **65** | ≥ 65 años |

Un residente pide al asistente de IA la función. El código se ejecuta sin errores. Pero con una paciente de 65 años y PAD de 60 mmHg, la función da 0 puntos cuando deberían ser 2: el asistente escribió `edad > 65` y `pad < 60`.

En el Tema 50 aprendimos a descubrir estos errores con casos de prueba. Hoy damos el paso siguiente: **devolver los fallos al modelo** para que corrija su propio código, y repetirlo hasta que todo funcione. Es el **ciclo de mejora recursiva**, la forma en que hoy se programa con IA.

## <font color='steelblue'>2. Conceptos clave (10 min)</font>

### <font color='steelblue'>2.1 El ciclo de mejora recursiva</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">El ciclo de mejora recursiva</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_090_ciclo.png" width="780" alt="El ciclo de mejora recursiva"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Especificar y escribir las pruebas <b>antes</b> de pedir el código.</li><li>Generar, ejecutar, verificar y corregir, hasta cumplir un criterio de parada.</li><li>El humano decide la especificación y cuándo el resultado es aceptable.</li></ul></div>

#### Desarrollo para el estudio

```
  ┌──────────────┐     ┌──────────────┐     ┌──────────────┐
  │ 1. GENERAR   │ ──▶ │ 2. EJECUTAR  │ ──▶ │ 3. VERIFICAR │
  │ el modelo    │     │ el código    │     │ con pruebas  │
  │ escribe      │     │              │     │              │
  └──────────────┘     └──────────────┘     └──────┬───────┘
         ▲                                         │
         │        ¿fallos? → 4. CORREGIR           │
         └──────── se devuelven los fallos ◀───────┘
                   al modelo

  Se detiene cuando se superan todas las pruebas o se agotan los intentos.
```

Lo que hace que el ciclo funcione no es el modelo, sino la **calidad del criterio de verificación**. Un modelo excelente con pruebas pobres produce código que parece correcto y no lo es.

### <font color='steelblue'>2.2 Primero la especificación y las pruebas</font>

El orden correcto, que en ingeniería del software se llama **desarrollo guiado por pruebas**:

1. Escribir la **especificación**: qué recibe la función, qué devuelve y con qué reglas.
2. Escribir los **casos de prueba** a partir de fuentes fiables, antes de ver ningún código.
3. Solo entonces pedir el código.

Si las pruebas se escriben después de ver el código, se tiende a escribir pruebas que ese código ya supera.

### <font color='steelblue'>2.3 Criterios de parada</font>

| Criterio | Para qué sirve |
| --- | --- |
| Todas las pruebas superadas | Objetivo del ciclo |
| Número máximo de intentos | Evita bucles infinitos y gasto de cuota |
| El modelo repite el mismo código | Señal de que no sabe corregirlo con la información dada |

### <font color='steelblue'>2.4 La ley de Goodhart</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">Ley de Goodhart: cuando la prueba se convierte en objetivo</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_090_goodhart.png" width="780" alt="Ley de Goodhart: cuando la prueba se convierte en objetivo"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Si se corrige mirando siempre las mismas pruebas, el código acaba ajustado a ellas.</li><li>El rendimiento real se mide con <b>pruebas reservadas</b>, que no se usaron para corregir.</li></ul></div>

#### Desarrollo para el estudio

*«Cuando una medida se convierte en objetivo, deja de ser una buena medida.»* Si el único objetivo del ciclo es superar las pruebas, el código acabará superándolas, pero no necesariamente será correcto en los casos que las pruebas no cubren. Por eso:

- Las pruebas deben cubrir los **valores frontera** de cada criterio.
- Conviene guardar algunas pruebas **reservadas** que el modelo no ve, para comprobar al final. Volveremos sobre esta idea en el Módulo 5.

### <font color='steelblue'>2.5 Seguridad</font>

Ejecutar código generado por IA es ejecutar código de un desconocido:

- **Leer el código antes de ejecutarlo**, aunque sea por encima.
- Ejecutarlo en un entorno aislado, como Colab, nunca en un ordenador con datos sensibles.
- No pedir código que acceda a ficheros, a internet o a Drive si la tarea no lo necesita.
- Nunca incluir datos de pacientes en la especificación ni en las pruebas.

## <font color='steelblue'>3. Demostración: un ciclo de mejora recursiva (20 min)</font>

### <font color='steelblue'>3.1 Conexión con el modelo</font>

Esta celda prepara la conexión con el modelo de lenguaje (Tema 80). Lee la clave de Secretos sin mostrarla. Si no hay clave, el cuaderno funciona con respuestas de ejemplo. Si el modelo indicado ya no existe, cambia la variable `MODELO`.

In [ ]:
%pip install -q google-genai

MODELO = "gemini-2.5-flash"

try:
    from google.colab import userdata
    from google import genai
    from google.genai import types
    cliente = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
    print("Conexión preparada con el modelo", MODELO)
except Exception as error:
    cliente = None
    print("Sin clave o fuera de Colab: se usarán respuestas de ejemplo.", type(error).__name__)


def preguntar(prompt, temperatura=0.2):
    """Envía un prompt al modelo y devuelve su texto (None si no hay conexión)."""
    if cliente is None:
        return None
    respuesta = cliente.models.generate_content(
        model=MODELO,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=temperatura),
    )
    return respuesta.text

### <font color='steelblue'>3.2 La especificación</font>

In [ ]:
ESPECIFICACION = """
Escribe en Python una función llamada curb65 con estos parámetros:
- confusion (bool): desorientación de nueva aparición
- urea_mmol_l (float): urea sérica en mmol/L
- fr_rpm (int): frecuencia respiratoria en respiraciones por minuto
- pas_mmHg (int): presión arterial sistólica
- pad_mmHg (int): presión arterial diastólica
- edad (int): edad en años

Devuelve un entero de 0 a 5 sumando un punto por cada criterio:
- confusión presente
- urea > 7 mmol/L
- frecuencia respiratoria >= 30
- presión sistólica < 90 o presión diastólica <= 60
- edad >= 65

Devuelve solo el código de la función, dentro de un bloque ```python```, sin ejemplos de uso.
"""

### <font color='steelblue'>3.3 Los casos de prueba, escritos antes que el código</font>

Cada caso indica los datos de entrada y el resultado esperado, calculado a mano. Hay casos para **cada criterio** y para **cada valor frontera**: 7 frente a 7,1 mmol/L, 29 frente a 30 rpm, 90 mmHg de PAS, 60 mmHg de PAD y 64 frente a 65 años.

In [ ]:
base = dict(confusion=False, urea_mmol_l=5.0, fr_rpm=18, pas_mmHg=130, pad_mmHg=80, edad=50)

PRUEBAS = [
    (base, 0),
    ({**base, "confusion": True}, 1),
    ({**base, "urea_mmol_l": 7.0}, 0),
    ({**base, "urea_mmol_l": 7.1}, 1),
    ({**base, "fr_rpm": 29}, 0),
    ({**base, "fr_rpm": 30}, 1),
    ({**base, "pas_mmHg": 90}, 0),
    ({**base, "pas_mmHg": 89}, 1),
    ({**base, "pad_mmHg": 61}, 0),
    ({**base, "pad_mmHg": 60}, 1),
    ({**base, "edad": 64}, 0),
    ({**base, "edad": 65}, 1),
    ({**base, "edad": 65, "pad_mmHg": 60}, 2),
    (dict(confusion=True, urea_mmol_l=9.0, fr_rpm=32, pas_mmHg=85, pad_mmHg=50, edad=80), 5),
]
print(len(PRUEBAS), "casos de prueba")

La sintaxis `{**base, "edad": 65}` crea una copia del diccionario `base` cambiando solo la edad.

### <font color='steelblue'>3.4 Ejecutar y verificar el código generado</font>

In [ ]:
def verificar(codigo):
    """Ejecuta el código en un espacio aislado y devuelve la lista de fallos."""
    espacio = {}
    try:
        exec(codigo, espacio)
        funcion = espacio["curb65"]
    except Exception as error:
        return [f"El código no se puede ejecutar: {type(error).__name__}: {error}"]

    fallos = []
    for entrada, esperado in PRUEBAS:
        try:
            obtenido = funcion(**entrada)
        except Exception as error:
            fallos.append(f"Con {entrada} se produce {type(error).__name__}")
            continue
        if obtenido != esperado:
            fallos.append(f"Con {entrada} se esperaba {esperado} y se obtuvo {obtenido}")
    return fallos

### <font color='steelblue'>3.5 Generar código: con el modelo o simulado</font>

Si hay conexión, se pide el código al modelo, añadiendo el código anterior y sus fallos cuando los hay. Sin conexión, se usan tres versiones simuladas que reproducen errores habituales de los asistentes.

In [ ]:
import re

VERSIONES_SIMULADAS = [
    """def curb65(confusion, urea_mmol_l, fr_rpm, pas_mmHg, pad_mmHg, edad):
    puntos = confusion + (urea_mmol_l > 7) + (fr_rpm >= 30)
    puntos += (pas_mmHg < 90 or pad_mmHg < 60) + (edad > 65)
    return puntos""",
    """def curb65(confusion, urea_mmol_l, fr_rpm, pas_mmHg, pad_mmHg, edad):
    puntos = confusion + (urea_mmol_l > 7) + (fr_rpm >= 30)
    puntos += (pas_mmHg < 90 or pad_mmHg < 60) + (edad >= 65)
    return puntos""",
    """def curb65(confusion, urea_mmol_l, fr_rpm, pas_mmHg, pad_mmHg, edad):
    puntos = int(confusion) + (urea_mmol_l > 7) + (fr_rpm >= 30)
    puntos += (pas_mmHg < 90 or pad_mmHg <= 60) + (edad >= 65)
    return int(puntos)""",
]


def generar(codigo_anterior, fallos, intento):
    """Devuelve el código de la función: del modelo o de las versiones simuladas."""
    if cliente is None:
        return VERSIONES_SIMULADAS[min(intento - 1, len(VERSIONES_SIMULADAS) - 1)]
    prompt = ESPECIFICACION
    if codigo_anterior:
        prompt += (
            "\nTu versión anterior era:\n```python\n" + codigo_anterior + "\n```\n"
            "Falla en estos casos de prueba:\n- " + "\n- ".join(fallos)
            + "\nCorrige la función."
        )
    texto = preguntar(prompt, temperatura=0.2)
    bloque = re.search(r"```(?:python)?\n(.*?)```", texto, re.DOTALL)
    return bloque.group(1).strip() if bloque else texto.strip()

### <font color='steelblue'>3.6 El ciclo completo</font>

In [ ]:
MAX_INTENTOS = 5
registro = []
codigo, fallos = None, []

for intento in range(1, MAX_INTENTOS + 1):
    codigo = generar(codigo, fallos, intento)
    fallos = verificar(codigo)
    registro.append({"intento": intento, "pruebas_falladas": len(fallos),
                     "pruebas_superadas": len(PRUEBAS) - len(fallos)})
    print(f"Intento {intento}: {len(PRUEBAS) - len(fallos)} de {len(PRUEBAS)} pruebas superadas")
    for fallo in fallos[:3]:
        print("   ·", fallo)
    if not fallos:
        print("\nTodas las pruebas superadas. Código final:\n")
        print(codigo)
        break
else:
    print("\nSe agotaron los intentos sin superar todas las pruebas: revisar a mano.")

La construcción `for … else` ejecuta el bloque `else` solo si el bucle termina **sin** pasar por `break`, es decir, si se agotan los intentos.

### <font color='steelblue'>3.7 El registro del ciclo</font>

Registrar cada iteración permite analizar el ciclo: cuántos intentos hicieron falta y cómo evolucionaron los fallos. En el Módulo 5 compararemos así, con métodos estadísticos, distintas formas de pedir código al modelo.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

tabla = pd.DataFrame(registro)
print(tabla.to_string(index=False))

plt.figure(figsize=(6, 3.5))
plt.plot(tabla["intento"], tabla["pruebas_superadas"], marker="o")
plt.axhline(len(PRUEBAS), linestyle="--", color="gray", label="Total de pruebas")
plt.xlabel("Intento")
plt.ylabel("Pruebas superadas")
plt.xticks(tabla["intento"])
plt.ylim(0, len(PRUEBAS) + 1)
plt.title("Evolución del ciclo de mejora recursiva")
plt.legend()
plt.show()

### <font color='steelblue'>3.8 Goodhart en acción: pruebas insuficientes</font>

¿Qué habría pasado con pruebas pobres, sin valores frontera? La primera versión simulada, con dos errores, las habría superado todas:

In [ ]:
PRUEBAS_COMPLETAS = PRUEBAS
PRUEBAS = [
    (base, 0),
    ({**base, "confusion": True}, 1),
    (dict(confusion=True, urea_mmol_l=9.0, fr_rpm=32, pas_mmHg=85, pad_mmHg=50, edad=80), 5),
]
print("Fallos con pruebas pobres:", len(verificar(VERSIONES_SIMULADAS[0])))

PRUEBAS = PRUEBAS_COMPLETAS
print("Fallos con pruebas completas:", len(verificar(VERSIONES_SIMULADAS[0])))

**Con pruebas pobres, el ciclo habría terminado en el primer intento con un código erróneo.** Una paciente de 65 años con PAD de 60 mmHg habría recibido 0 puntos en lugar de 2. La calidad del ciclo es la calidad de sus pruebas.

### <font color='steelblue'>Ampliación: pruebas reservadas</font>

Una forma de protegerse frente a Goodhart es separar las pruebas en dos grupos: las que se devuelven al modelo durante el ciclo y unas **pruebas reservadas** que solo se usan al final. Si el código supera las primeras pero falla en las reservadas, el ciclo ha aprendido a superar el examen, no a resolver el problema.

In [ ]:
PRUEBAS_RESERVADAS = [
    (dict(confusion=False, urea_mmol_l=7.5, fr_rpm=30, pas_mmHg=88, pad_mmHg=58, edad=70), 4),
    (dict(confusion=True, urea_mmol_l=6.9, fr_rpm=22, pas_mmHg=95, pad_mmHg=62, edad=40), 1),
]

espacio = {}
exec(codigo, espacio)
for entrada, esperado in PRUEBAS_RESERVADAS:
    obtenido = espacio["curb65"](**entrada)
    print("correcto" if obtenido == esperado else "FALLO", "·", entrada, "→", obtenido)

### <font color='steelblue'>Ampliación: errores frecuentes</font>

**Escribir las pruebas después de ver el código:** se acaban escribiendo pruebas que ese código ya supera.

**No poner límite de intentos:** el ciclo puede gastar toda la cuota de la API.

**Ejecutar el código generado sin leerlo.**

**Devolver al modelo solo «está mal»:** sin indicar qué casos fallan y por qué, el modelo no tiene información para corregir.

**Dar por correcta una función porque supera las pruebas:** solo es correcta en lo que las pruebas comprueban.

## <font color='steelblue'>4. Práctica en la app Bioestadística (10 min)</font>

Abre la **[unidad 90 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)** y completa sus misiones:

1. **Misión básica (sin asistente):** identificar qué valores frontera faltan en un conjunto de casos de prueba.
2. **Misión clínica (con pistas):** escribir la especificación y las pruebas de la escala de Wells para trombosis venosa profunda.
3. **Misión reto (con asistente):** ejecutar el ciclo de mejora recursiva para esa escala, registrar las iteraciones y comprobar el resultado con pruebas reservadas.

Al terminar, exporta el **informe de la unidad**.

## <font color='steelblue'>5. Autoevaluación (5 min)</font>

Responde sin asistente y sin ejecutar código.

In [ ]:
#@title Pregunta interactiva
preguntas = [{'question': '¿Cuál es el orden correcto para programar con un asistente de IA?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Especificación, casos de prueba, generación del código y verificación',
               'correct': True,
               'feedback': 'Correcto: las pruebas se escriben antes de ver el código.'},
              {'answer': 'Generación del código, casos de prueba y verificación',
               'correct': False,
               'feedback': 'Si las pruebas se escriben después, se adaptan al código que ya '
                           'existe.'},
              {'answer': 'Generación del código y uso directo si no da errores',
               'correct': False,
               'feedback': 'Ejecutarse sin errores no significa ser correcto.'},
              {'answer': 'Casos de prueba, uso directo y especificación',
               'correct': False,
               'feedback': 'La especificación es lo primero: define qué se quiere.'}]},
 {'question': 'Una función CURB-65 usa edad > 65. ¿Qué caso de prueba descubre el error?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Un paciente de exactamente 65 años',
               'correct': True,
               'feedback': 'Correcto: el valor frontera es el que distingue > de >=.'},
              {'answer': 'Un paciente de 80 años',
               'correct': False,
               'feedback': 'Con 80 años, > 65 y >= 65 dan el mismo resultado.'},
              {'answer': 'Un paciente de 40 años',
               'correct': False,
               'feedback': 'Con 40 años, ninguna de las dos versiones suma el punto.'},
              {'answer': 'Ningún caso puede descubrirlo',
               'correct': False,
               'feedback': 'El caso frontera lo descubre.'}]},
 {'question': '¿Qué información conviene devolver al modelo cuando su código falla?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Los casos concretos que fallan, con el resultado esperado y el obtenido',
               'correct': True,
               'feedback': 'Correcto: es la información que permite corregir.'},
              {'answer': 'Solo «está mal, inténtalo otra vez»',
               'correct': False,
               'feedback': 'Sin saber qué falla, el modelo no tiene en qué basarse.'},
              {'answer': 'Las pruebas reservadas',
               'correct': False,
               'feedback': 'Las reservadas no se enseñan al modelo: sirven para la comprobación '
                           'final.'},
              {'answer': 'Los datos de un paciente real con el que falló',
               'correct': False,
               'feedback': 'Nunca se envían datos de pacientes.'}]},
 {'question': 'Un ciclo de mejora recursiva supera todas sus pruebas en el primer intento. ¿Qué '
              'puedes concluir?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Que el código es correcto en lo que comprueban las pruebas, y nada más',
               'correct': True,
               'feedback': 'Correcto: la garantía depende de la calidad de las pruebas.'},
              {'answer': 'Que el código es correcto en cualquier caso',
               'correct': False,
               'feedback': 'Las pruebas pobres del apartado 3.8 dejaban pasar dos errores.'},
              {'answer': 'Que el modelo no necesitaba pruebas',
               'correct': False,
               'feedback': 'Sin pruebas no habría forma de saberlo.'},
              {'answer': 'Que hay que aumentar la temperatura',
               'correct': False,
               'feedback': 'La temperatura no tiene que ver con la validez de las pruebas.'}]},
 {'question': '¿Por qué se fija un número máximo de intentos?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Para que el ciclo termine aunque el modelo no consiga corregir el código',
               'correct': True,
               'feedback': 'Correcto: evita bucles infinitos y gasto de cuota.'},
              {'answer': 'Porque el modelo empeora a partir del tercer intento',
               'correct': False,
               'feedback': 'No hay una regla fija; depende del problema.'},
              {'answer': 'Porque Python no permite más de cinco iteraciones',
               'correct': False,
               'feedback': 'Python no impone ese límite.'},
              {'answer': 'Para que el código sea más rápido',
               'correct': False,
               'feedback': 'El límite no afecta a la velocidad del código final.'}]}]
display_quiz(preguntas)

## <font color='steelblue'>Ideas clave</font>

- El ciclo de mejora recursiva genera código, lo ejecuta, lo verifica con pruebas y devuelve los fallos al modelo hasta que se superan o se agotan los intentos.
- La especificación y las pruebas se escriben **antes** del código, con atención a los valores frontera.
- Superar las pruebas solo garantiza lo que las pruebas comprueban (ley de Goodhart); las pruebas reservadas ayudan a detectarlo.
- El código generado por IA se lee antes de ejecutarlo, en un entorno aislado y sin datos de pacientes.

## <font color='steelblue'>Glosario</font>

| Término | Definición |
| --- | --- |
| Mejora recursiva | Ciclo en el que un modelo corrige su propia salida a partir de una verificación externa |
| Especificación | Descripción precisa de lo que debe hacer una función |
| Desarrollo guiado por pruebas | Forma de programar en la que las pruebas se escriben antes que el código |
| Criterio de parada | Condición que detiene un ciclo iterativo |
| Ley de Goodhart | Cuando una medida se convierte en objetivo, deja de ser una buena medida |
| Pruebas reservadas | Casos que no se usan durante el ciclo y sirven para la comprobación final |
| `exec()` | Función de Python que ejecuta código contenido en una cadena de texto |

## <font color='steelblue'>Para saber más</font>

- Lim WS, *et al.* Defining community acquired pneumonia severity on presentation to hospital: an international derivation and validation study. *Thorax*. 2003;58(5):377-382.
- [Tutorial de Python: errores y excepciones](https://docs.python.org/es/3/tutorial/errors.html)

---
*Tema 90 · Bioestadística (1964) · Grado en Medicina · Universidad Miguel Hernández de Elche · Fernando Borrás Rocher · CC BY 4.0*